### CRAB

In [ ]:
import sys
import matplotlib.pyplot as plt
import numpy as np
import os
from qutip import *
plt.rcParams.update({"text.usetex": True, "font.family": "serif", "font.size": 14})
src_path = os.path.abspath(os.path.join("..", "src"))
sys.path.append(src_path)

from crab_optimizer import CRABOptimizer

dim = 30
g = 0.3
omega_c = 1.0
omega_bases = [2.0] 
sigma = 0.7 
tau_s = np.pi / (2 * g)
T = 20 * tau_s
Nt = 400
tlist = np.linspace(0, T, Nt)
num_freqs = 20 

a = destroy(dim)
n_op = a.dag() * a
n_tot = tensor(n_op, qeye(2))
identity = tensor(qeye(dim), qeye(2))
x_couple = tensor(a + a.dag(), sigmax())
sz_tot = tensor(qeye(dim), sigmaz())
sx_tot = tensor(qeye(dim), sigmax())

H0 = omega_c * (n_tot + (1/2) * identity) + g * x_couple 
H1 = -0.5 * sz_tot

psi0 = tensor(basis(dim,0), basis(2,0))

n_target = 6
target_fock = basis(dim, n_target)

def db_to_r(db):
    return db / (20 * np.log10(np.e))
r = 3 / (20 * np.log10(np.e))  # parametro di squeezing
theta = np.pi/4  # angolo di squeezing (opzionale)
target_squeezed = squeeze(dim, r * np.exp(1j * theta)) * basis(dim, 0)

alpha = 2.0
cat_a   = coherent(dim, alpha)
cat_ma  = coherent(dim, -alpha)
cat_ia  = coherent(dim, 1j*alpha)
cat_mia = coherent(dim, -1j*alpha)
C_plus  = (cat_a + cat_ma).unit()
C_minus = (cat_a - cat_ma).unit()
Ci_plus  = (cat_ia + cat_mia).unit()
Ci_minus = (cat_ia - cat_mia).unit()
target_cat = (C_plus - Ci_plus).unit()

target_state = target_fock #target_squeezed #target_cat 

def fidelity_cost(final_state, pulses):
    field_state = ptrace(final_state, 0)
    return 1 - fidelity(target_state, field_state)

def fidelity_cost_penalty(final_state, pulses):
    field_state = ptrace(final_state, 0)
    lambda_neg = 1e-1
    k = 50.0 

    penalty_total = 0
    for i in range(len(pulses)):
        pulse_neg = np.minimum(0, pulses[i])  
        penalty_total += np.sum(np.log1p(np.exp(-k * pulse_neg)) / k)

    infidelity = 1 - fidelity(target_state, field_state)
    return infidelity + lambda_neg * penalty_total

def fidelity_cost_penalty_amp(final_state, pulses):

    field_state = ptrace(final_state, 0)
    infidelity = 1 - fidelity(target_state, field_state)

    # u_min = 0.3
    # u_max = 2.5
    # lam = 1e-2
    # penalty = 0
    # for pulse in pulses:

    #     penalty_low = np.maximum(0, u_min - pulse)**2
    #     penalty_high = np.maximum(0, pulse - u_max)**2

    #     penalty += np.sum(penalty_low + penalty_high)

    dt = tlist[1] - tlist[0]
    E_target = 500.0   
    lam_energy = 1e-2

    E = 0
    for pulse in pulses:
        E += np.sum(pulse**2) * dt

    penalty_energy = (E - E_target)**2

    return infidelity + lam_energy * penalty_energy # + lam * penalty

def fidelity_cost_penalty_der(final_state, pulses):

    field_state = ptrace(final_state, 0)
    infidelity = 1 - fidelity(target_state, field_state)

    lam_amp = 1e-2     # penalità ampiezza 
    lam_der = 1e-3     # penalità derivata 

    u_min = 0.1
    u_max = 2.5

    penalty_amp = 0
    penalty_der = 0

    dt = tlist[1] - tlist[0]

    for pulse in pulses:

        # --- Penalità ampiezza ---
        penalty_low = np.maximum(0, u_min - pulse)**2
        # penalty_high = np.maximum(0, pulse - u_max)**2
        penalty_amp += np.sum(penalty_low) #+ penalty_high)

        # --- Penalità derivata ---
        du = np.diff(pulse) / dt
        penalty_der += np.sum(du**2)

    return infidelity + lam_der * penalty_der #+ lam_amp * penalty_amp


optimizer = CRABOptimizer(
    H_drift=H0,
    H_controls=[H1],    
    T=T,
    num_tslots=Nt,
    num_freqs=num_freqs,
    omega_bases=omega_bases,
    freq_range=(0, 3*omega_c),
    basis_type="uniform",
    )
    # basis_type="fourier")

result = optimizer.optimize(
    psi0=psi0,
    cost_function=fidelity_cost,
    max_iter=None,
    method="Nelder-Mead",
    # clip_neg = True,
    # val_clip_neg = 0.0,
    # clip_pos = True,
    # val_clip_pos = 3
)


##############################################################################################
# P = 3
# class CRABFixedPower(CRABOptimizer):
#     def _build_single_control(self, params_block, freqs):

#         norm = np.linalg.norm(params_block)
#         if norm == 0:
#             scaled = params_block
#         else:
#             scaled = np.sqrt(P) * params_block / norm

#         coeffs_a = scaled[:self.num_freqs]
#         coeffs_b = scaled[self.num_freqs:]

#         pulse = np.array([
#             self._basis(t, coeffs_a, coeffs_b, freqs)
#             * self._boundary_function(t)
#             for t in self.tlist
#         ])

#         return pulse

# optimizer = CRABFixedPower(
#     H_drift=H0,
#     H_controls=[H1],
#     T=T,
#     num_tslots=Nt,
#     num_freqs=num_freqs,
#     freq_range=(0, 2*omega_c),
#     basis_type="uniform",
#     seed=16
# )

# result = optimizer.optimize(
#     psi0=psi0,
#     cost_function=fidelity_cost,
#     max_iter=None,
#     method="Nelder-Mead"
# )
###############################################################################

print("Fidelity finale:", fidelity(target_state, ptrace(result.final_state, 0)))

control_opt = result.control_pulses[0]

H_total = [H0, [H1, control_opt]]
result_evol = sesolve(H_total, psi0, tlist)

# control_sz = result.control_pulses[0]  # controllo su sigma_z
# control_sx = result.control_pulses[1]  # controllo su sigma_x

# H_total = [H0, [sz_tot, control_sz], [sx_tot, control_sx]]
# result_evol = sesolve(H_total, psi0, tlist)

states = result_evol.states

n_expect = [expect(n_tot, psi) for psi in states]

dt = tlist[1] - tlist[0]
E = np.sum(control_opt**2)*dt 
print(f"Energy of the control pulse: {E:.2f}")

plt.figure(figsize=(6,4))
plt.plot(tlist/tau_s, n_expect, 'crimson', lw=2)
plt.xlabel("$t / \\tau_s$")
plt.ylabel("$\\langle n \\rangle$")
plt.grid(True)
# plt.savefig("../results/meeting_13032026/lagrange/sigma_x/T20/nmedio_05_35.png", dpi=300)
plt.savefig('n_mean.pdf', dpi=300)
plt.show()

plt.figure(figsize=(6,4))
plt.plot(tlist/tau_s, control_opt, 'darkgreen', lw=2)
# plt.plot(tlist/tau_s, control_sz, 'darkgreen', lw=2)
plt.xlabel("$t / \\tau_s$")
plt.ylabel("$\\omega_q(t)$")
plt.grid(True)
# plt.savefig("../results/meeting_13032026/lagrange/sigma_x/T20/drive_05_35.png", dpi=300)
plt.savefig('drive.pdf', dpi=300)
plt.show()
# plt.figure(figsize=(6,4))
# plt.plot(tlist/tau_s, control_sx, 'darkgreen', lw=2)
# plt.xlabel("$t / \\tau_s$")
# plt.ylabel("$\\epsilon(t)$")
# plt.grid(True)
# plt.show()
 
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
w_target = wigner(target_state, np.linspace(-5,5,200), np.linspace(-5,5,200))
plt.contourf(np.linspace(-5,5,200), np.linspace(-5,5,200), w_target, 100, cmap='RdBu_r')
plt.colorbar()
plt.title("Wigner target state")

plt.subplot(1,2,2)
final_state_field = ptrace(states[-1], 0)  # tracing over qubit
w_final = wigner(final_state_field, np.linspace(-5,5,200), np.linspace(-5,5,200))
plt.contourf(np.linspace(-5,5,200), np.linspace(-5,5,200), w_final, 100, cmap='RdBu_r')
plt.colorbar()
plt.title("Wigner optimized state")
plt.tight_layout()
plt.savefig('wigner.pdf', dpi=300)
# plt.savefig("../results/meeting_13032026/lagrange/sigma_x/T20/wigner_05_35.png", dpi=300)
plt.show()

dt = tlist[1] - tlist[0]

control = control_opt - np.mean(control_opt)

fft_vals = np.fft.fft(control)
freqs = np.fft.fftfreq(len(control), d=dt)

mask = freqs >= 0

plt.figure(figsize=(6,4))
plt.plot(freqs[mask] * (2*np.pi), np.abs(fft_vals[mask]))

plt.axvline(2*omega_c, color='red', linestyle='--', label=r'$2\omega_c$')

# plt.xlim(0, 5)
plt.xlabel("Frequency")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)
plt.show()


# import pickle
# with open("optimized_pulse_squeezed.pkl", "wb") as f:
#     pickle.dump(control_opt, f)

### GRAPE

In [ ]:
import sys
import os
import matplotlib.pyplot as plt
import numpy as np
from qutip import *
import pickle
plt.rcParams.update({"text.usetex": False, "font.family": "serif", "font.size": 14})

# Path assoluto della cartella data
data_path = os.path.abspath(os.path.join("..", "data"))
# with open(os.path.join(data_path, "control_opt_crab.pkl"), "rb") as f:
    # control_opt = pickle.load(f)
# Path assoluto della cartella src
src_path = os.path.abspath(os.path.join("..", "src"))
sys.path.append(src_path)


import pandas as pd

out_dir = "results/crab_fock_scaling_20260507_110941" # g = 0.3

df = pd.read_csv(f"{out_dir}/data.csv")

n_targets = sorted(df["n_target"].unique())

best_pulses = {}

for n in n_targets:

    subset = df[df["n_target"] == n]

    idx_best = subset["fidelity"].idxmax()

    best_pulses[n] = df.loc[idx_best, "control"]


from grape_optimizer import GRAPEOptimizer, GRAPEResult
dim = 30
g = 0.3
omega_base = 2.0
omega_c = 1.0
tau_s = np.pi / (2 * g)
T = 20 * tau_s
Nt = 300
tlist = np.linspace(0, T, Nt)

a = destroy(dim)
n_op = a.dag() * a
n_tot = tensor(n_op, qeye(2))
identity = tensor(qeye(dim), qeye(2))
x_couple = tensor(a + a.dag(), sigmax())
sz_tot = tensor(qeye(dim), sigmaz())
sx_tot = tensor(qeye(dim), sigmax())

H0 = omega_c * (n_tot + 0.5 * identity) + g * x_couple
H1 = -0.5 * sz_tot
# H0 = omega_c * (n_tot + (1/2) * identity) + g * x_couple - 0.5 * omega_c * sz_tot
# H1 = sx_tot

psi0 = ket2dm(tensor(basis(dim,0), basis(2,0)))


n_target = 1


rho_osc = ket2dm(basis(dim, n_target))  
rho_qub = ket2dm(basis(2,1))
target_fock = tensor(rho_osc, qeye(2))
# target_fock = tensor(rho_osc, rho_qub)
target_fock /= target_fock.tr()

r = 3 / (20 * np.log10(np.e))  # parametro di squeezing
theta = np.pi/4  # angolo di squeezing 
squeezed_state = squeeze(dim, r * np.exp(1j * theta)) * basis(dim, 0)  # stato squeezed puro
rho_squeezed = ket2dm(squeezed_state)

target_squeezed = tensor(rho_squeezed, qeye(2))
target_squeezed /= target_squeezed.tr()

alpha = 2.0
cat_a   = coherent(dim, alpha)
cat_ma  = coherent(dim, -alpha)

cat_ia  = coherent(dim, 1j*alpha)
cat_mia = coherent(dim, -1j*alpha)

C_plus  = (cat_a + cat_ma).unit()
C_minus = (cat_a - cat_ma).unit()
Ci_plus  = (cat_ia + cat_mia).unit()
Ci_minus = (cat_ia - cat_mia).unit()

rho_cat = (C_plus - 1j*Ci_plus).unit()
target_cat = ket2dm(
    tensor(
        (C_plus - 1j*Ci_plus).unit(),
        basis(2,0)
    ))

target_state = target_fock #target_squeezed #target_cat 


control_opt = best_pulses[n_target]
control_opt = np.fromstring(
    control_opt.strip("[]"),
    sep=" "
)
plt.figure(figsize=(6,4))
plt.plot(control_opt, 'darkgreen', lw=2)
plt.xlabel("$t / \\tau_s$")
plt.ylabel("$\\omega_q(t)$")
plt.grid(True)
plt.show()
# control_opt_fin = [control_opt[1:-1].copy()]


def gaussian_guess(tlist, omega0=1.0, amp=0.2, sigma=None):
    T = tlist[-1]
    if sigma is None:
        sigma = T / 5
    pulse = omega0 + amp * np.exp(-(tlist - T/2)**2 / (2 * sigma**2))
    return pulse
initial_pulse = gaussian_guess(tlist, omega0=1.0, amp=0.2)
initial_guess = [initial_pulse]


optimizer = GRAPEOptimizer(
    H_drift=H0,
    H_controls=[H1],  
    T=T,
    num_tslots=Nt,
    omega_bases=[omega_base],  
    epsilon=1e-1,      
    seed=None
)

# result: GRAPEResult = optimizer.optimize_gradient_ascent(
#     psi0=psi0,
#     target_state=target_fock,
#     max_iter=1000,
#     goal=0.99
# )

result1: GRAPEResult = optimizer.optimize_minimize(
    psi0=psi0,
    target_state=target_state,
    max_iter=np.inf,
    amp_bound=(0.8, 2.2),
    # initial_guess=control_opt_fin
    initial_guess=[control_opt]#[pulse]#initial_guess
)
control_opt1 = result1.control_pulses[0]  
states1 = optimizer.forward_propagation(result1.control_pulses, psi0)
F = fidelity(ptrace(states1[-1], 0), ptrace(target_state, 0))
print(F)
n_expect = [expect(n_tot, psi) for psi in states1]



dt = T / Nt

cost = (1/2) * np.sum((control - omega_bases[0])**2) * dt



plt.figure(figsize=(6,4))
plt.plot(tlist/tau_s, n_expect[1:], 'crimson', lw=2)
plt.xlabel("$t / \\tau_s$")
plt.ylabel("$\\langle n \\rangle$")
plt.grid(True)
# plt.savefig("../results/meeting_13032026/grape/nmedio_07_13.png", dpi=300)
plt.show()

plt.figure(figsize=(6,4))
plt.step(tlist/tau_s, control_opt1, where='post', color='darkgreen', lw=2)
plt.xlabel("$t / \\tau_s$")
plt.ylabel("$\\omega_q(t)$")
plt.grid(True)
# plt.savefig("../results/meeting_13032026/grape/drive_07_13.png", dpi=300)
plt.show()

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
w_target = wigner(ptrace(target_state, 0), np.linspace(-5,5,200), np.linspace(-5,5,200))
plt.contourf(np.linspace(-5,5,200), np.linspace(-5,5,200), w_target, 100, cmap='RdBu_r')
plt.colorbar()
plt.title("Target state Wigner")

plt.subplot(1,2,2)
final_state_field = ptrace(states1[-1], 0)
w_final = wigner(final_state_field, np.linspace(-5,5,200), np.linspace(-5,5,200))
plt.contourf(np.linspace(-5,5,200), np.linspace(-5,5,200), w_final, 100, cmap='RdBu_r')
plt.colorbar()
plt.title("Optimized state Wigner")
plt.tight_layout()
# plt.savefig("../results/meeting_13032026/grape/wigner_07_13.png", dpi=300)
plt.show()


dt = tlist[1] - tlist[0]

control = control_opt1 - np.mean(control_opt1)

fft_vals = np.fft.fft(control)
freqs = np.fft.fftfreq(len(control), d=dt)

mask = freqs >= 0

plt.figure(figsize=(6,4))
plt.plot(freqs[mask] * (2*np.pi), np.abs(fft_vals[mask]))

plt.axvline(2*omega_c, color='red', linestyle='--', label=r'$2\omega_c$')

# plt.xlim(0, 5)
plt.xlabel("Frequency")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)
plt.show()

### Interpolation GRAPE

In [ ]:
from scipy.interpolate import PchipInterpolator

def smooth_interpolate(tlist, control):
    cs = PchipInterpolator(tlist, control)
    return cs

tlist_fine = np.linspace(0, T, 2*Nt)  # più punti
cs = smooth_interpolate(tlist, control_opt1)

control_smooth = cs(tlist_fine)

optimizer_fine = GRAPEOptimizer(
    H_drift=H0,
    H_controls=[H1],
    T=T,
    num_tslots=len(tlist_fine),
    omega_bases=[omega_base],
    epsilon=1e-1,
    seed=42
)

states_smooth = optimizer_fine.forward_propagation([control_smooth], psi0)

# import pickle
# with open("optimized_pulse_squeezed1.pkl", "wb") as f:
#     pickle.dump(control_smooth, f)

F = fidelity(ptrace(states_smooth[-1], 0), ptrace(target_state, 0))
print(F)
n_expect = [expect(n_tot, psi) for psi in states_smooth]

plt.figure(figsize=(6,4))
plt.plot(tlist_fine/tau_s, n_expect[1:], 'crimson', lw=2)
plt.xlabel("$t / \\tau_s$")
plt.ylabel("$\\langle n \\rangle$")
plt.grid(True)
plt.tight_layout()
# plt.savefig('n_mean.pdf', dpi=300)
# plt.savefig("../results/meeting_13032026/grape/nmedio_07_13.png", dpi=300)
plt.show()

plt.figure(figsize=(6,4))
plt.plot(tlist_fine/tau_s, control_smooth, color='darkgreen', lw=2)
plt.xlabel("$t / \\tau_s$")
plt.ylabel("$\\omega_q/\omega_c$")
plt.grid(True)
plt.tight_layout()
# plt.savefig('drive.pdf')
# plt.savefig("../results/meeting_13032026/grape/drive_07_13.png", dpi=300)
plt.show()

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
w_target = wigner(ptrace(target_state, 0), np.linspace(-5,5,200), np.linspace(-5,5,200))
plt.contourf(np.linspace(-5,5,200), np.linspace(-5,5,200), w_target, 100, cmap='RdBu_r')
plt.colorbar()
plt.title("Target state Wigner")
    
plt.subplot(1,2,2)
final_state_field = ptrace(states_smooth[-1], 0)
w_final = wigner(final_state_field, np.linspace(-5,5,200), np.linspace(-5,5,200))
plt.contourf(np.linspace(-5,5,200), np.linspace(-5,5,200), w_final, 100, cmap='RdBu_r')
plt.colorbar()
plt.title("Optimized state Wigner")
plt.tight_layout()
# plt.savefig('wigner.pdf', dpi=300)
# plt.savefig("../results/meeting_13032026/grape/wigner_07_13.png", dpi=300)
plt.show()

dt = tlist[1] - tlist[0]

control = control_smooth - np.mean(control_smooth)

fft_vals = np.fft.fft(control)
freqs = np.fft.fftfreq(len(control), d=dt)

mask = freqs >= 0

plt.figure(figsize=(6,4))
plt.plot(freqs[mask] * (2*np.pi), np.abs(fft_vals[mask]))
plt.axvline(2*omega_c, color='red', linestyle='--', label=r'$2\omega_c$')
# plt.xlim(0, 5)
plt.xlabel("Frequency")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)
plt.show()
